# 一：RAG

---

## 1.1 RAG（检索增强生成）

**定义**：引入外部知识库，让大模型在回答前先在这个外部知识库中检索，再回答

### 优点 ✅

- 回答准确，检索库可以更新，所以回答的依据也可以更新
- 可解释性强，可以在检索库核实准确性
- 缓解幻觉问题（避免胡说八道）
- 在本地，隐私性好

### RAG 对比 模型微调

| 特性 | RAG | 模型微调 |
|------|-----|----------|
| 知识更新 | 动态更新 | 需要重新训练 |
| 定制化程度 | 低 | 高 |
| 数据处理需求 | 低 | 高 |
| 幻觉问题 | 缓解 | 减少 |
| 计算资源 | 需要检索 | 需要训练 |
| 可追溯性 | 可追溯 | 难追溯 |

---

## 1.1.1 基本步骤

分为三大步骤：**建库 -> 检索 -> 生成**（`indexing` -> `retrieval` -> `generation`）

### 建库（Indexing）🔧

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | 收集数据 | 收集原始数据，转化为统一文本格式 |
| 2 | 文本分割 | 将文本分割成适当大小的块（chunk） |
| 3 | 向量化 | 使用预训练模型将文本转化为向量 |
| 4 | 存储 | 将向量存入向量数据库，构建索引 |

### 检索（Retrieval）🔍

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | Query Translation | 查询改写：Multi Query、RAG Fusion、Decomposition、Step Back、HyDE |
| 2 | Routing | 路由：根据 query 分类，选择正确的数据源 |
| 3 | Query Construction | 查询构造：NER、关键词提取，转为数据库查询参数 |
| 4 | 向量检索 | 计算相似度，找出最相关的文档 |
| 5 | 排序/重排 | 根据相似度排序，优化结果顺序 |
| 6 | 过滤 | 过滤无关或低质量结果 |

### 生成（Generation）📝

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | 构建Prompt | 将 query 和检索到的文档结合 |
| 2 | LLM生成 | 基于文档生成答案 |
| 3 | 后处理 | 置信度评估、格式解析、多候选筛选 |

---

## 1.1.2 Query Translation（查询改写）🔄

**属于检索阶段**，将 question 变成更容易检索的形式，提升后续检索的效果

---

### 1. Multi Query（多查询）

将 question 从**多个角度**改写，每一个 question 都进行检索，查询结果取**并集**

- 可以获得更全面的文档集合
- 缺点：可能引入过多不相关内容

---

### 2. RAG Fusion（RAG 融合）

对于检索到的文档做**相互排名**，通过加权各个 question 在文档中的查询结果，得到**综合排名**

- 取前几名作为最终结果
- 比 Multi Query 更精确

---

### 3. Decomposition（查询分解）

将原始 query 拆解成**多个子 query**，每个子 query 都进行检索

| 方式 | 说明 |
|------|------|
| **顺序依赖** | 每个子问题会影响后续子问题的提问和解答过程，类似局部推理 |
| **独立并行** | 每个子问题互不影响，最后合并答案 |

---

### 4. Step Back（后退）

从一个**具体问题**出发，通过 **few-shot**，生成一个更高层次、更抽象的问题

- 抽象问题有助于检索到更相关的基础概念
- 适用于：原始问题过于细节，难以直接检索的情况

---

### 5. HyDE（Hypothetical Document Embeddings）📄

**假设文档嵌入**，核心思想：

1. 利用已有的大模型生成一个**假设性回答**
2. 将生成的假设文档输入编码器生成**嵌入向量**
3. 用这个向量去找**语义相似**的真实文档

**优势**：假设文档中包含相关关键词，可以检索到更多相关内容

---

## 总结：Query Translation 方法对比

| 方法 | 核心思想 | 适用场景 |
|------|----------|----------|
| Multi Query | 多角度改写 | 需要全面覆盖 |
| RAG Fusion | 综合排名 | 平衡全面与精确 |
| Decomposition | 拆解子问题 | 复杂多部分问题 |
| Step Back | 抽象化 | 问题过于细节 |
| HyDE | 假设文档 | 关键词匹配困难 |

---

## 1.1.3 Routing（路由）🎯

**属于检索阶段**，本质是一个**分类任务**，根据 question 判断去哪个数据源检索

---

### 实现方式

| 方式 | 说明 | 优点 | 缺点 |
|------|------|------|------|
| **基于规则** | 用预定义关键词匹配 | 简单快速 | 关键词覆盖不全 |
| **基于机器学习** | 收集样本，微调模型 | 准确率高 | 需要训练数据 |
| **基于提示词工程** | 用大模型判断意图 | 零成本/少样本 | 依赖模型能力 |

---

### 路由类型

| 类型 | 说明 |
|------|------|
| **Logical Routing** | 将 query 路由到最相关的资源库 |
| **Semantic Routing** | 用 embeddings 找最相关的 prompt 或文档 |

---

## 1.1.4 Query Construction（查询构造）🔧

**属于检索阶段**，把用户的**自然语言**转换成**数据库能理解的查询参数**

---

### 核心思想

利用**命名实体识别（NER）**等技术，将自然语言 question 转成合适的搜索和过滤参数

---

### 示例

**用户 query（自然语言）**：
```
how to use multi-modal models in an agent, only videos under 5 minutes
```

**转换成搜索参数**：

| 参数 | 值 | 含义 |
|------|-----|------|
| content_search | multi-modal models agent | 搜索内容 |
| title_search | multi-modal models agent | 搜索标题 |
| max_length_sec | 300 | 只找 5 分钟以下的视频 |

---

### 适用场景

- SQL 数据库查询
- Elasticsearch 搜索
- API 参数构造
- 任何需要结构化查询的场景

---


1.1.5 Indexing ：将文档拆成向量形式，建立索引
